pip install 하려면 <br>
현재 분석 환경 경로 -> 터미널 열기 -> 가상환경 실행 -> uv pip install ~~

In [1]:
# import pandas as pd
import polars as pl

# ① 데이터 정제 + 평가셋 라벨링

**목표:** 1~2점 + 코멘트 있는 리뷰를 정제하고, 사람이 검수한 정답셋(`eval.csv`)과 예시 풀(`pool.csv`)을 만든다.

---

## 1. 정제
- [ ] 6개 테이블 조인 (reviews, orders, order_items, products + translation, sellers, customers)
- [ ] 대상 필터링: 1~2점 & 코멘트 있음
- [ ] 중복 제거: `(review_id, order_id)` 기준
- [ ] 판매자 귀속 규칙 결정 (단일 판매자 주문만 / 관련 판매자 전원 귀속)
- [ ] 텍스트 정리: 제목 + 본문 합치기, 공백 정리 (포르투갈어 원문 유지)

## 2. 라벨 체계 정의 ⭐ 가장 중요
- [ ] 카테고리 확정 (예: 배송지연 / 미수령 / 오배송·누락 / 품질·파손 / 설명과 다름 / 응대·환불 / 기타)
- [ ] 카테고리별 정의, 경계 규칙, 애매한 사례 문서화
- [ ] 단일 라벨(주 원인 하나)로 시작

## 3. 라벨링
- [ ] 파일럿 50건 → 가이드라인 수정
- [ ] 평가셋 300~500건 (층화 샘플, 성능 측정 전용)
- [ ] 예시 풀 200~500건 (few-shot / RAG용) — **평가셋과 절대 겹치지 않게**
- [ ] 일관성 점검: 일부를 며칠 뒤 재라벨링 → kappa 확인
- [ ] 배송지연 라벨을 실제 배송일과 대조
      (`order_delivered_customer_date > order_estimated_delivery_date`)

---

## 주의사항
- 코멘트 없는 리뷰가 절반 이상
- `review_id`는 유일하지 않음
- 한 주문에 판매자가 여러 명일 수 있음
- 평가셋은 LLM 선라벨링 금지 → 사람이 검수한 라벨 기준

## 산출물
- [ ] 정제 parquet
- [ ] 라벨 가이드 문서
- [ ] `eval.csv`
- [ ] `pool.csv`

# EDA

- 데이터 구조

<!-- ![데이터 구조](data/data_structure.png) -->
<img src="data/data_structure.png" width="800">

In [3]:
df = pl.read_csv("data/raw/olist_order_reviews_dataset.csv")
df.head()

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
str,str,i64,str,str,str,str
"""7bc2406110b926393aa56f80a40eba…","""73fc7af87114b39712e6da79b0a377…",4,null,null,"""2018-01-18 00:00:00""","""2018-01-18 21:46:59"""
"""80e641a11e56f04c1ad469d5645fdf…","""a548910a1c6147796b98fdf73dbeba…",5,null,null,"""2018-03-10 00:00:00""","""2018-03-11 03:05:13"""
"""228ce5500dc1d8e020d8d1322874b6…","""f9e4b658b201a9f2ecdecbb34bed03…",5,null,null,"""2018-02-17 00:00:00""","""2018-02-18 14:36:24"""
"""e64fb393e7b32834bb789ff8bb3075…","""658677c97b385a9be170737859d351…",5,null,"""Recebi bem antes do prazo esti…","""2017-04-21 00:00:00""","""2017-04-21 22:02:06"""
"""f7c4243c7fe1938f181bec41a392bd…","""8e6bfb81e283fa7e4f11123a3fb894…",5,null,"""Parabéns lojas lannister adore…","""2018-03-01 00:00:00""","""2018-03-02 10:26:53"""


In [ ]:
# 활용법
df = pl.read_csv(
    "data.csv",
    separator=",",           # 구분자 (pandas의 sep)
    has_header=True,         # 헤더 여부
    columns=["a", "b"],      # 필요한 컬럼만
    schema_overrides={"id": pl.Utf8},  # 특정 컬럼 타입 지정
    encoding="utf8",         # 한글 CP949 파일이면 "cp949"는 미지원 → 아래 참고
)

## lazy로 불러오기
lf = pl.scan_csv("data.csv")   # LazyFrame — 아직 안 읽음
df = lf.filter(pl.col("a") > 10).select("a", "b").collect()  # 이때 실제로 읽음

# 정제

- [ ] 6개 테이블 조인 (reviews, orders, order_items, products + translation, sellers, customers)
- [ ] 대상 필터링: 1~2점 & 코멘트 있음
- [ ] 중복 제거: `(review_id, order_id)` 기준
- [ ] 판매자 귀속 규칙 결정 (단일 판매자 주문만 / 관련 판매자 전원 귀속)
- [ ] 텍스트 정리: 제목 + 본문 합치기, 공백 정리 (포르투갈어 원문 유지)

# 라벨 체계 정의

- [ ] 카테고리 확정 (예: 배송지연 / 미수령 / 오배송·누락 / 품질·파손 / 설명과 다름 / 응대·환불 / 기타)
- [ ] 카테고리별 정의, 경계 규칙, 애매한 사례 문서화
- [ ] 단일 라벨(주 원인 하나)로 시작

# 라벨링

- [ ] 파일럿 50건 → 가이드라인 수정
- [ ] 평가셋 300~500건 (층화 샘플, 성능 측정 전용)
- [ ] 예시 풀 200~500건 (few-shot / RAG용) — **평가셋과 절대 겹치지 않게**
- [ ] 일관성 점검: 일부를 며칠 뒤 재라벨링 → kappa 확인
- [ ] 배송지연 라벨을 실제 배송일과 대조
      (`order_delivered_customer_date > order_estimated_delivery_date`)